In [66]:
import numpy as np
import imutils
import cv2 as cv
import glob 

In [67]:
images = glob.glob('./data/rotated/*.png')

# corner ids  
CORNER_IDS = {"top_left": 0, "top_right": 1, "bottom_right": 2, "bottom_left": 3}

# detector and dict
aruco_dict = cv.aruco.getPredefinedDictionary(cv.aruco.DICT_4X4_50)
aruco_params = cv.aruco.DetectorParameters()
detector = cv.aruco.ArucoDetector(aruco_dict, aruco_params)

In [68]:
# image prep 
def load_image(path):
    # load
    image = cv.imread(path)
    
    if image is None:
        raise FileNotFoundError(f"Could not read {path}")

    # resize, grayscale
    image = imutils.resize(image, width=1000)
    image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    
    return image

In [69]:
# grab vertices of img 
def find_corners(img):

    # detect markers
    corners, ids, _ = detector.detectMarkers(img)
    if ids is None:
        raise RuntimeError("No ArUco markers detected.")

    # map each id to its vertices
    markers = {int(mid): c.reshape((4, 2)) for c, mid in zip(corners, ids)}
    
    # grab outer vertex
    outer = {name: markers[mid][mid] for name, mid in CORNER_IDS.items()}

    return outer["top_left"], outer["bottom_left"], outer["bottom_right"], outer["top_right"]

In [70]:
# reorient img w/ edges 
def transform(img, coords):
    # top left, bottom left, bottom right, top right 
    tl, bl, br, tr = coords   

    # set new output size  
    width = int(max(np.linalg.norm(tl - tr), np.linalg.norm(bl - br)))
    height = int(max(np.linalg.norm(tl - bl), np.linalg.norm(tr - br)))

    # original and new vertices 
    input_pts = np.float32([tl, bl, br, tr])
    output_pts = np.float32([[0, 0], [0, height - 1],
                             [width - 1, height - 1], [width - 1, 0]])

    # transform
    M = cv.getPerspectiveTransform(input_pts, output_pts)
    corrected = cv.warpPerspective(img, M, (width, height), flags=cv.INTER_LINEAR)

    # resize to original width
    return imutils.resize(corrected, width=1000)

In [71]:
def prepare(path):
    image = load_image(path)
    corners = find_corners(image)
    return transform(image, corners)

In [72]:
for image in images:
    okey = cv.imwrite(f'./output/{image.replace('./data/rotated/','')}', prepare(image))

    print(f"{image} okey" if okey else f"{image} no okey")

./data/rotated/test-1.png okey
./data/rotated/test-2.png okey
./data/rotated/test-3.png okey
./data/rotated/test-4.png okey
